# Parameters, State, and Memory

The previous section built models as trees of modules and found every
parameter by walking the tree. So far the framework has handled those
tensors for us, and we never had to ask what else a model holds or how much
memory training it takes. For a small model there is no need to ask. For a
large one both questions become pressing. Consider a model with 8 billion
parameters, the size of the Llama 3 model that
that section mentioned. Its weights take 32 GB in the
32-bit floating-point format we have used so far. Training it takes four
times as much memory, 128 GB, before it has seen a single example: every
weight needs a gradient of the same size, and the Adam optimizer, which we
meet in that section, keeps two more tensors of that size, running
averages of the gradients and of their squares. That is more than an 80 GB
GPU holds.

To arrive at such numbers we need to know every tensor a model holds and
what each one is for. Quite ordinary operations need the same knowledge: a
checkpoint must save every tensor the computation depends on, a move to a
GPU must take all of them along, and fine-tuning must decide which of them
the optimizer may change.

Frameworks make this possible by *registration*
(that section): each layer announces the tensors it owns
to the module system, which files them by name in the tree of modules, and
the optimizer, the checkpoint, and the device move then all walk the same
tree. This section is an inventory of that tree. Its first part asks what a
model holds: the tensors the optimizer trains, and those it must leave
alone. Its second part asks what they cost, and counts the bytes that
training stores per parameter. The third part is closer to a reference. It
collects three things that are routinely done to parameters (sharing one
between two layers, excluding some from training, and averaging them over
time), each of which borrows a model or a method from a later chapter. It
can be skimmed now and consulted when those chapters call for it.

In [ ]:
import tensorflow as tf

## The State Tree

### Access by Path

Our running example is a residual MLP: an input layer maps 20 features to
64, two residual blocks of width 64 follow, and an output layer produces 10
outputs. The blocks are the residual block of
that section, defined again here so that this section
stands on its own: each adds the output of a small two-layer network, its
*body*, to its input.

In [ ]:
class ResidualBlock(tf.keras.layers.Layer):
    def __init__(self, num_hiddens, **kwargs):
        super().__init__(**kwargs)
        self.body = tf.keras.Sequential([
            tf.keras.layers.Dense(num_hiddens, activation='relu'),
            tf.keras.layers.Dense(num_hiddens)])

    def call(self, X):
        return X + self.body(X)

net = tf.keras.Sequential([tf.keras.layers.Dense(64), ResidualBlock(64),
                           ResidualBlock(64), tf.keras.layers.Dense(10)])
X = tf.random.normal((2, 20))
net(X).shape

The parameters are the leaves of the module tree, and the path of names that
leads to a leaf (the figure) is also the way to reach it in code,
which suits debugging a single layer. The optimizer and the checkpoint,
however, need every leaf. To this end each framework offers a traversal of
the whole tree that returns the parameters together with their paths.

In Keras it is `net.weights`, a list of variables that each carry their
path. Variables do not store their gradients: a `tf.GradientTape` returns
them as a second list, parallel to the variables.

The model has 12 parameter tensors. We print their number and the four
that belong to the first residual block:

In [ ]:
len(net.weights), [(v.path, v.shape) for v in net.weights[2:6]]

The path `sequential_2/residual_block/sequential/dense_1/kernel` runs from the
outer `Sequential` through the first residual block and its `body` (a
`Sequential` of its own) to the first `Dense` layer inside it and the leaf
`kernel`; in code it reads `net.layers[1].body.layers[0].kernel`. Note that the
segments are layer *names*, assigned at construction: a default derived from
the class, plus a counter that keeps names unique within the program (this is
the third `Sequential` created, hence `sequential_2`). Default names therefore
depend on the order of construction, and a tool that must match paths across
separately written programs should name the layers explicitly. (Keras
checkpoints do not rely on them: they restore by the structure of the saved
objects, that section.) Keras splits the same list by trainability,
and so far the split is trivial:

In [ ]:
[v.path for v in net.weights] == [v.path for v in net.trainable_weights]

Every consumer walks this one tree under these names. The equality above
holds only because all of this model's state is trainable. That need not be
so.

### Parameters and Buffers

Suppose a model standardizes its inputs with a mean and a standard deviation
computed once from training data. These numbers must be saved with the model,
since predictions are wrong without them, and they must move to the GPU with
it. Yet the optimizer must never change them: they are statistics of the
data, not quantities to fit. A plain attribute would not do, since no
traversal sees it (that section). Batch normalization
[@Ioffe.Szegedy.2015], a layer that we study in
that section, keeps tensors of the same kind: a running mean and
variance of its inputs, updated during training and used at prediction time.

Such a persistent tensor without a gradient is commonly called a *buffer*,
and every framework has a way to register one so that checkpoints and device
moves see it while the optimizer does not. This gives a rule that covers
every tensor a layer touches: make it a *parameter* if the optimizer should
update it, a *buffer* if checkpoints need it, and a local variable if it
lives for one call. The last kind includes the intermediate values that
backpropagation stores during a training step (that section). They
belong to the computation, not to the model, and are neither saved nor
registered. Plain Python attributes hold architecture constants such as
layer widths.

Here is a layer, `Standardizer`, that standardizes its inputs with statistics
computed from a reference sample: one mean and one standard deviation per
feature.

Keras calls every registered tensor a *weight* and distinguishes by a flag:
`add_weight(trainable=False)` creates a variable that is saved with the
model but never handed to the optimizer. This is how `BatchNormalization`
stores its running statistics.

In [ ]:
class Standardizer(tf.keras.layers.Layer):
    def __init__(self, mean, std, **kwargs):
        super().__init__(**kwargs)
        self.mean = self.add_weight(
            shape=mean.shape, trainable=False, name='mean',
            initializer=tf.keras.initializers.Constant(mean))
        self.std = self.add_weight(
            shape=std.shape, trainable=False, name='std',
            initializer=tf.keras.initializers.Constant(std))
        self.out = tf.keras.layers.Dense(2)

    def call(self, X):
        return self.out((X - self.mean) / self.std)

sample = tf.random.normal((100, 4)) * tf.range(1., 5.)
standardizer = Standardizer(tf.reduce_mean(sample, 0),
                            tf.math.reduce_std(sample, 0))
_ = standardizer(sample[:2])   # build the Dense layer
[v.path for v in standardizer.weights]

All four variables are weights, so all four are checkpointed. A training
step differentiates and updates `trainable_weights` only, and the two
statistics sit in the other half of the split:

In [ ]:
([v.path for v in standardizer.trainable_weights],
 [v.path for v in standardizer.non_trainable_weights])

Registration decides more than what is saved. Operations on the model as a
whole, such as a change of dtype or a move to another device, reach the
registered tensors and no others (that section). We can
watch this without a GPU.

A tensor stored as a plain attribute is not counted, not saved, and absent
from every traversal. Attach one, and the list of weights does not grow:

In [ ]:
standardizer.note = tf.zeros(4)   # plain attribute, not registered
[v.path.split('/')[-1] for v in standardizer.weights]

Device placement works differently in TensorFlow: a variable's device is
decided when the variable is *created*, not by a later move, so there is no
`.to()` that could leave a tensor behind. With a GPU visible, Keras creates
variables on it by default, non-trainable weights included. Each variable
reports its placement:

In [ ]:
standardizer.mean.value.device   # decided when the variable was created

## Training Memory

### Bytes per Parameter

How much memory does training our residual MLP take? For the parameters and
what the optimizer stores about them, the answer follows from three counts
per parameter: the weight itself, its gradient, and the optimizer's state
for it. Minibatch gradient descent, which we have used so far, keeps no
state. In 32-bit floating point (fp32), 4 bytes per number, it stores 8
bytes per parameter. Adam keeps two more numbers per parameter, the running
averages of the gradient and of its square. We count for Adam, since almost
all large models are trained with it or with one of its variants, so this
is the count that matters in practice:

| Training state                       | Precision | Bytes per parameter |
|--------------------------------------|-----------|---------------------|
| Weights                              | fp32      | 4                   |
| Gradients                            | fp32      | 4                   |
| Adam's average of the gradients      | fp32      | 4                   |
| Adam's average of their squares      | fp32      | 4                   |
| Total                                |           | 16                  |

These 16 bytes are a floor. They omit the *activations*, the intermediate
values that backpropagation keeps from the forward pass, whose size grows
with the batch; they also omit temporary workspaces and the memory the
allocator holds in reserve. that section measures all of these on a
GPU. For our residual MLP, the floor is:

In [ ]:
n = net.count_params()
weights = sum(int(tf.size(v)) * tf.as_dtype(v.dtype).size
              for v in net.weights)   # fp32: 4 bytes
grads, adam_state = weights, 2 * weights
print(f'{n} parameters: {(weights + grads + adam_state) / 2**20:.2f} MiB '
      'for weights + gradients + Adam state')

The count checks by hand: the input layer has $20 \times 64 + 64 = 1344$
parameters, each residual block $2 \times (64 \times 64 + 64) = 8320$, and
the output layer $64 \times 10 + 10 = 650$, for a total of 18,634. Adam's
state is real memory, allocated tensor by tensor. After one training step,
its two running averages are two more copies of the model:

In [ ]:
adam = tf.keras.optimizers.Adam()
with tf.GradientTape() as tape:
    loss = tf.reduce_sum(net(X))
grads = tape.gradient(loss, net.trainable_weights)
# the first step allocates the moments
adam.apply_gradients(zip(grads, net.trainable_weights))
moments = sum(int(tf.size(v)) for v in adam.variables
              if v.ndim > 0)   # skip step count and learning rate
moments == 2 * n

For our network the floor is 0.28 MiB, which is why memory has not
mattered so far. For the 8-billion-parameter model the same arithmetic gives
the 128 GB of the opening, whatever the batch size. One might hope that
computing with 16-bit numbers, the *mixed-precision* training of
that section, halves this figure. Unfortunately it does not lower
the floor at all: an fp32 copy of the weights and fp32 running averages are
kept, so that small updates are not rounded away
[@Micikevicius.Narang.Alben.ea.2018], and the 16-bit copies come on
top.

### Counting Without Allocating

Every count so far read the shapes of tensors that already existed. For the
8B model this order is backwards: to learn whether its 32 GB of weights fit
in memory, we would first have to allocate them. Luckily, the count needs
only the shape and the dtype of each parameter, and both follow from the
arguments of the constructors. Take the output layer of the 8B model, which
maps a hidden state of width 4,096 to one score for each of the 128,256
*tokens*, the words and pieces of words, that the model can output. Its
weight has $4096 \times 128{,}256 = 525{,}336{,}576$ entries, 2.1 GB in
fp32. We create the layer without its data and count.

Keras creates the variables of a layer when it builds the layer, at its
first call or through an explicit `build`, and each variable holds its data
from the moment it is created. An unbuilt layer holds no variables, so we
compute the count from the shapes: the kernel of a `Dense` layer has the
shape (inputs, units).

In [ ]:
head = tf.keras.layers.Dense(128256, use_bias=False)   # not built
n_head = 4096 * head.units   # kernel shape: (inputs, units)
print(head.weights, f'{n_head:,} parameters, '
      f'{4 * n_head / 1e9:.1f} GB in fp32')

The list of weights is empty. Calling the layer on an input of width 4,096,
or calling `head.build((None, 4096))`, creates the kernel and allocates the
2.1 GB.

Applied to every layer of the 8B model, the same count returns its 8 billion
parameters, and with them the memory figures of the opening, without
allocating a byte.

## Shared, Frozen, and Averaged Parameters

### Tied Parameters

One tensor can serve several roles in a model. The standard example is a
language model, which reads a sequence of tokens from a vocabulary $V$ and
predicts the next one (that section). Its first layer is an
*embedding*, a $|V| \times d$ table whose row $v$ is the $d$-dimensional
vector that represents token $v$. Its output layer maps a $d$-dimensional
hidden state to $|V|$ scores, one per token, so its weight matrix has the
same shape and an analogous meaning: one vector per token. Why keep two
tables? *Tying* the two roles to a single tensor saves $|V| \times d$
parameters, and the studies that introduced it found that it also improves
the predictions [@Press.Wolf.2017; @Inan.Khosravi.Socher.2017]. This is
at least plausible: words that can replace one another in a sentence should
be similar as inputs and receive similar scores as outputs, and tying leaves
the model only one notion of similarity to learn. The savings are large. In
GPT-2 small [@Radford.Wu.Child.ea.2019], the shared
$50257 \times 768$ table holds about 38.6 million of the model's 124 million
parameters, roughly 31%.

![One weight matrix W, two call sites: the embedding looks up a row by token id, the output head multiplies by its transpose to produce logits. Tying means both point at the same tensor, so gradients from both uses sum into one place.](https://raw.githubusercontent.com/smolix/d2l-neu/notebooks/img/bg-weight-tying.svg)

the figure shows the two call sites. A toy language model,
`ToyLM`, with a vocabulary of 100 tokens and $d = 16$, shows the mechanics.

Keras has no tying flag, so we write it in code: a small head layer that
creates no variables of its own and instead multiplies by the transpose of
the embedding table it is handed. One variable, two call sites:

In [ ]:
class TiedHead(tf.keras.layers.Layer):
    def __init__(self, emb, **kwargs):
        super().__init__(**kwargs)
        self.emb = emb   # reuse the embedding layer; create nothing

    def call(self, h):
        return tf.matmul(h, self.emb.embeddings, transpose_b=True)

class ToyLM(tf.keras.Model):
    def __init__(self, vocab_size, d, tied=True, **kwargs):
        super().__init__(**kwargs)
        self.emb = tf.keras.layers.Embedding(vocab_size, d)
        self.body = tf.keras.layers.Dense(d)
        self.head = (TiedHead(self.emb) if tied else
                     tf.keras.layers.Dense(vocab_size, use_bias=False))

    def call(self, tokens):
        return self.head(tf.nn.relu(self.body(self.emb(tokens))))

lm = ToyLM(vocab_size=100, d=16)
tokens = tf.random.uniform((2, 8), 0, 100, dtype=tf.int32)
_ = lm(tokens)   # build
[(v.path, tuple(v.shape)) for v in lm.weights]

Is the shared table counted once or twice? We compare the tied model with an
untied twin.

Because the head created no variable, there is no aliasing to keep track of:
the traversal reports the table once, and no head matrix exists.

In [ ]:
untied = ToyLM(vocab_size=100, d=16, tied=False)
_ = untied(tokens)
(lm.count_params(), untied.count_params())

The tied model has $1600 + 272 = 1872$ parameters: the table and the body
layer. The untied one adds a second $100 \times 16$ table, 3472 in all.

A checkpoint of the tied model stores the table once, because the head owns
nothing to store. In exchange, tied and untied checkpoints differ in
structure: the untied model carries one more kernel, so loading one into the
other is a mismatch that we must resolve explicitly.

What happens to the gradients? During backpropagation each use of the tensor
contributes a gradient, and the contributions add up in the gradient of the
one shared tensor. Let $\mathbf{W}_\textrm{emb}$ and $\mathbf{W}_\textrm{head}$
be the two tables of the untied model. If both hold the values of the tied
table $\mathbf{W}$, the two models compute the same function, and the chain
rule gives

$$\frac{\partial L}{\partial \mathbf{W}} =
\frac{\partial L}{\partial \mathbf{W}_\textrm{emb}} +
\frac{\partial L}{\partial \mathbf{W}_\textrm{head}}.$$

The tied gradient is the sum of the untied gradients. We check this by
loading the tied model's values into the untied one and running the same
backward pass through both:

In [ ]:
untied.set_weights(lm.get_weights()   # tied values, head kernel materialized
                   + [tf.transpose(lm.emb.embeddings)])
with tf.GradientTape() as tape:
    loss = tf.reduce_sum(lm(tokens))
g_tied = tape.gradient(loss, lm.emb.embeddings)
with tf.GradientTape() as tape:
    loss = tf.reduce_sum(untied(tokens))
g_emb, g_head = tape.gradient(
    loss, [untied.emb.embeddings, untied.head.kernel])
g_emb = tf.convert_to_tensor(g_emb)   # densify the embedding-lookup gradient
bool(tf.reduce_max(tf.abs(g_tied - (g_emb + tf.transpose(g_head)))) < 1e-5)

### Frozen Parameters

Fine-tuning adapts a model trained on one task to another
(that section). A common recipe keeps the pretrained layers
fixed and trains only a new output layer, which needs little data and little
memory. Keeping a parameter fixed is called *freezing* it. We freeze
everything but the output layer of a fresh copy of our residual MLP, which
should leave 650 of its 18,634 parameters trainable.

In Keras, freezing is a flag on the layer: setting `layer.trainable = False`
moves the layer's variables from `trainable_weights` to
`non_trainable_weights`, the split a training step consults. (Models trained
with `compile` and `fit` see the change only at the next `compile`; the
manual steps below read the split directly.)

In [ ]:
finetune = tf.keras.Sequential([tf.keras.layers.Dense(64), ResidualBlock(64),
                                ResidualBlock(64), tf.keras.layers.Dense(10)])
_ = finetune(X)   # build
for layer in finetune.layers[:-1]:
    layer.trainable = False

(sum(int(tf.size(v)) for v in finetune.trainable_weights),
 finetune.count_params())

One Adam step should now move the head and nothing else. We compare every
parameter before and after it.

The tape differentiates with respect to the variables we hand it, and
`trainable_weights` now holds only the head.

In [ ]:
head_opt = tf.keras.optimizers.Adam(learning_rate=0.1)
before = [tf.identity(v) for v in finetune.weights]
with tf.GradientTape() as tape:
    loss = tf.reduce_sum(finetune(X))
head_opt.apply_gradients(zip(tape.gradient(loss, finetune.trainable_weights),
                             finetune.trainable_weights))
[not bool(tf.reduce_all(b == v)) for b, v in zip(before, finetune.weights)]

Only the last two entries, the head's kernel and bias, changed. A Keras
optimizer allocates its state at the first step for the variables it is
handed, so `head_opt` holds Adam's running averages for 650 parameters.

Freezing has two limits, and neither raises an error. The first is merely
wasteful: a freeze does not reclaim optimizer state that already exists. An
optimizer that was built over all parameters and has taken a step keeps
their running averages, 8 bytes per parameter, whatever is frozen
afterwards. For our 18,634 parameters nobody will mind. The last exercise
works out what the same oversight means for the 8B model.

The second limit matters more. Freezing stops gradients, and some state
changes without one. Batch normalization (that section) updates
its running mean and variance in the *forward pass*, whenever the layer runs
in training mode. No optimizer is involved, so there is no reason why
freezing the layer's parameters should hold its statistics still. We pass
one batch through such a layer in training mode, without any optimizer
step, and ask whether the running mean is unchanged.

Keras treats this case specially. Setting `trainable = False` on a
`BatchNormalization` layer also switches its forward pass to inference
mode, so the statistics stay fixed even when the layer is called with
`training=True`:

In [ ]:
bn = tf.keras.layers.BatchNormalization(momentum=0.9, epsilon=1e-5)
_ = bn(tf.zeros((8, 4)), training=True)   # build, then freeze
bn.trainable = False
before = tf.identity(bn.moving_mean)
_ = bn(tf.random.normal((8, 4)) + 3, training=True)
bool(tf.reduce_all(bn.moving_mean == before))

The running mean stayed where it was, but this exception holds for
`BatchNormalization` only. Everywhere else, `trainable` and the `training`
call argument are independent switches: the first governs which variables
receive updates, the second the behaviors of training mode, and a frozen
dropout layer still drops units when called with `training=True`. To keep
any other layer with state fixed during fine-tuning, set both.

### Weight Averages

A last kind of state is derived from the parameters but not trained: an
*exponential moving average* (EMA) of the weights, kept next to the trained
ones and used for evaluation. After every optimizer step, the average
$\bar{\mathbf{w}}$ moves a fraction of the way toward the current weights
$\mathbf{w}$,

$$\bar{\mathbf{w}} \leftarrow 0.9\, \bar{\mathbf{w}} + 0.1\, \mathbf{w},$$

so it lags behind the iterates and smooths out their fluctuations. Whether
it evaluates better than the last iterate is a question for
that section, which measures it. All that concerns us here
is that the average is state: like BatchNorm statistics, it changes outside
backpropagation and belongs in the checkpoint. We start the average at the
current weights and continue the fine-tuning above for five steps.

Keras optimizers can keep this average themselves (`use_ema=True`). Written
out, it is one variable per trainable weight and one `assign` after each
step:

In [ ]:
ema = [tf.Variable(v) for v in finetune.trainable_weights]
for _ in range(5):
    with tf.GradientTape() as tape:
        loss = tf.reduce_sum(finetune(X))
    grads = tape.gradient(loss, finetune.trainable_weights)
    head_opt.apply_gradients(zip(grads, finetune.trainable_weights))
    for avg, v in zip(ema, finetune.trainable_weights):
        avg.assign(0.9 * avg + 0.1 * v)
float(tf.reduce_max(tf.abs(ema[-1] - finetune.layers[-1].bias)))

The printed number is the gap between the averaged and the current bias of
the head, and it is 0.3686 on every run. This is no accident, and we can
predict it. The loss is the sum of the outputs, so the gradient of each bias
entry is the batch size, 2, at every step. Adam divides its average of the
gradient by the square root of its average of the squared gradient, so a
gradient that never changes moves each entry by the learning rate, 0.1, per
step. If the weights move by 0.1 per
step, the gap $e$ between the average and the weights obeys
$e \leftarrow 0.9\,(e + 0.1)$, which starts at 0 and reaches $0.3686$ after
five steps. The average carries a memory of earlier weights that the weights
themselves have lost, so a checkpoint that dropped it could not resume the
average where it stopped.

Every parameter we met in this section, whether tied, frozen, or averaged,
started from a value that some rule chose. The next section asks which rule
to choose, and how to apply it over the tree.

## Summary

A model's state is a tree of named tensors, and registration decides which
consumers see each one: the optimizer sees the parameters, while checkpoints
and device moves also see buffers, persistent tensors such as running
statistics that receive no gradient. Unregistered tensors are left behind.
Training with Adam in fp32 stores 16 bytes per parameter before any
activation: weight, gradient, and Adam's two running averages. For an
8-billion-parameter
model that is 128 GB. Tying gives two layers one tensor, which is counted
and updated once and receives the sum of the gradients of its uses.
Freezing keeps parameters from changing, but it does not free optimizer
state that already exists, and it does not stop the updates of running
statistics in training mode.

## Exercises

1. [code] **Running statistics.** This problem needs only one fact about
   the batch normalization layer of that section, the update
   below. In training mode the layer normalizes with the minibatch mean
   $\hat{\boldsymbol{\mu}}_{\mathcal{B}}$ and variance, and each training
   step updates the running mean $\bar{\boldsymbol{\mu}}$ by

   $$\bar{\boldsymbol{\mu}} \leftarrow \rho\, \bar{\boldsymbol{\mu}} + (1 - \rho)\, \hat{\boldsymbol{\mu}}_{\mathcal{B}}.$$

    1. A fresh layer starts from $\bar{\boldsymbol{\mu}} = \mathbf{0}$.
       Predict $\bar{\boldsymbol{\mu}}$ after the layer has processed $k$
       batches in training mode, each with minibatch mean 3 in every
       feature. Check the prediction for $k = 1, 5, 20$ on a fresh layer
       over 4 features, without any optimizer step, and read off the value
       of $\rho$ the layer uses by default.
    1. Suppose you registered $\bar{\boldsymbol{\mu}}$ as a trainable
       parameter and removed the update above. What gradient does the
       training loss send to $\bar{\boldsymbol{\mu}}$? How do the training
       loss and the evaluation accuracy then compare with those of an
       unmodified model?
    1. Find a loss in $\bar{\boldsymbol{\mu}}$ alone whose gradient step
       with learning rate $\eta$ on each minibatch reproduces the removed
       update, and state the $\eta$ that matches $\rho$.

1. **Parameters and buffers.** that section separates
   trained parameters from persistent state that receives no gradient.
   State a one-line rule that classifies every entry of a model's state as
   one or the other, and check it against the state of `standardizer` in
   that section and against the batch normalization
   layer `bn` of that section, including any integer counter
   it keeps.

1. [code] **Memory footprint.** Write a helper that reports, for each
   top-level block of `net`, its number of state entries and its bytes
   under fp32 Adam training with every parameter trained.
    1. Which block dominates? At what output width would the output layer
       overtake one residual block?
    1. Insert a batch normalization layer over the 64 hidden features
       between the two residual blocks. Extend the helper so that it costs
       the layer's running statistics correctly, and report each block's
       bytes per entry. For which blocks is the share of the total bytes
       larger than the share of the total entries, and for which is it
       smaller? Explain both directions.

4. [code] **Tied parameters.** Perturb `untied.head.kernel` so that the two
   tables of the untied twin no longer agree. Then build a tied `ToyLM`
   and fill it from `untied` with `set_weights`. Which values must the
   conversion discard or combine? Compare the logits of the converted model
   on `tokens` with those of `untied`, and state when the converted model
   reproduces `untied` exactly.

5. [code] **Tying and freezing.** Consider the tied `ToyLM` of this
   section.
    1. Set `lm.emb.trainable = False` but leave `lm.head` alone. How many
       trainable parameters remain, and can the head still adapt? What does
       this show about the interaction between tying and freezing?
    1. Starting from the values of `lm`, build a model whose embedding stays
       fixed while its head adapts, and verify both properties after one
       optimizer step. How many parameters does it store and how many does
       it train, compared with a fully trainable `lm`?

6. **Fine-tuning memory.** A language model has 8 billion parameters. Its
   output layer is a $4096 \times 128{,}256$ matrix, not tied to the
   embedding. All state is fp32, and the optimizer is Adam.
    1. How many bytes do weights, gradients, and Adam's running averages take when
       every parameter is trained? How many 80 GB GPUs does that state
       alone fill?
    1. Freeze everything except the output layer, and build the optimizer
       over the output layer only. How many bytes does the persistent state
       take now, and does it fit on one 80 GB GPU?
    1. Suppose instead that an optimizer built over all parameters took
       one step before the freeze and is kept afterwards. How many bytes
       remain allocated, and does the state still fit?

[Discussions](https://d2l.discourse.group/t/36753)